# Notebook 01: Data Exploration & Preprocessing
## Why Did My Model Predict That? — Heart Disease Explainability

**Project**: MasterSchool AI Enhanced Productivity — Project 3  
**Dataset**: UCI Heart Disease Dataset (303 records, 14 features)  
**Goal**: Load, explore, and preprocess the data for model training

### Objectives
1. Load the UCI Heart Disease dataset
2. Perform exploratory data analysis (EDA)
3. Visualize feature distributions and correlations
4. Handle missing values and encode features
5. Split data into train/test sets
6. Save preprocessed data for downstream notebooks

In [0]:
%pip install ucimlrepo pandas numpy matplotlib seaborn scikit-learn xgboost shap lime pickle5 --quiet

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import pickle
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Libraries imported successfully")

In [0]:
# Load dataset from UCI ML Repository
try:
    from ucimlrepo import fetch_ucirepo
        heart_disease = fetch_ucirepo(id=45)
            df = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)
                df.columns = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','num']
                    print("✅ Dataset loaded from UCI ML Repository")
                    except Exception as e:
                            print(f"Fallback to manual URL: {e}")
                                url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
                                    cols = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','num']
                                        df = pd.read_csv(url, names=cols, na_values='?')
                                            print("✅ Dataset loaded from URL")

                                            print(f"\nDataset shape: {df.shape}")
                                            print(f"Target distribution:\n{df['num'].value_counts()}")
                                            df.head()

In [0]:
# EDA: Basic statistics, missing values, target distribution
print("=== Dataset Info ===")
print(df.dtypes)
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nBasic statistics:")
display(df.describe())

# Target distribution
print(f"\n=== Target Variable (num) ===")
print(f"Original values: {sorted(df['num'].unique())}")
print("Binarizing: 0=No Disease, 1=Heart Disease (values 1-4 → 1)")

In [0]:
# EDA Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Heart Disease Dataset — EDA Overview', fontsize=16, fontweight='bold')

# 1. Target distribution
df_plot = df.copy()
df_plot['target'] = (df_plot['num'] > 0).astype(int)
axes[0,0].bar(['No Disease (0)', 'Heart Disease (1)'], 
              df_plot['target'].value_counts().sort_index().values,
                            color=['#2ecc71', '#e74c3c'], alpha=0.8)
                            axes[0,0].set_title('Target Variable Distribution')
                            axes[0,0].set_ylabel('Count')

                            # 2. Age by target
                            df_plot.boxplot(column='age', by='target', ax=axes[0,1], 
                                            boxprops=dict(color='steelblue'))
                                            axes[0,1].set_title('Age Distribution by Heart Disease')
                                            axes[0,1].set_xlabel('Target (0=No Disease, 1=Disease)')
                                            
                                            # 3. Correlation heatmap
                                            corr_cols = ['age','trestbps','chol','thalach','oldpeak']
                                            corr = df_plot[corr_cols + ['target']].corr()
                                            sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', 
                                                        ax=axes[1,0], cbar_kws={'shrink': 0.8})
                                                        axes[1,0].set_title('Feature Correlation Heatmap')
                                                        
                                                        # 4. Chest pain type
                                                        cp_counts = df_plot.groupby(['cp', 'target']).size().unstack(fill_value=0)
                                                        cp_counts.plot(kind='bar', ax=axes[1,1], color=['#2ecc71', '#e74c3c'], alpha=0.8)
                                                        axes[1,1].set_title('Chest Pain Type vs Heart Disease')
                                                        axes[1,1].set_xlabel('Chest Pain Type (1-4)')
                                                        axes[1,1].legend(['No Disease', 'Disease'])
                                                        axes[1,1].tick_params(axis='x', rotation=0)
                                                        
                                                        plt.tight_layout()
                                                        plt.show()
                                                        print("✅ EDA visualizations complete"))

In [0]:
# Preprocessing Pipeline
# Step 1: Binarize target
df['target'] = (df['num'] > 0).astype(int)
df = df.drop(columns=['num'])

# Step 2: Handle missing values
feature_cols = [c for c in df.columns if c != 'target']
imputer = SimpleImputer(strategy='median')
df[feature_cols] = imputer.fit_transform(df[feature_cols])

# Step 3: Feature scaling
scaler = StandardScaler()
X = df[feature_cols].values
y = df['target'].values
X_scaled = scaler.fit_transform(X)

# Step 4: Train/test split (80/20 stratified)
X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
        )

        print(f"✅ Preprocessing complete!")
        print(f"Training set: {X_train.shape[0]} samples")
        print(f"Test set: {X_test.shape[0]} samples")
        print(f"Features: {feature_cols}")
        print(f"Class balance (train): {dict(zip(*np.unique(y_train, return_counts=True)))}")

        # Step 5: Save preprocessed data
        data_dict = {
                'X_train': X_train, 'X_test': X_test,
                    'y_train': y_train, 'y_test': y_test,
                        'feature_names': feature_cols,
                            'scaler': scaler, 'imputer': imputer
                            }
                            with open('/tmp/preprocessed_data.pkl', 'wb') as f:
                                    pickle.dump(data_dict, f)
                                    print("✅ Data saved to /tmp/preprocessed_data.pkl")
        }
)